In [1]:
import datetime as dt

import talib
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from securities_load.securities.postgresql_database_functions import sqlalchemy_engine
from securities_load.securities.securities_table_functions import (
    retrieve_ohlcv_from_to,
)
from sqlalchemy.sql.elements import True_

In [2]:
load_dotenv()
engine = sqlalchemy_engine()
exchange_code = "XNAS"
ticker = "GOOG"
num_of_years = 10
start_date = dt.datetime.now() - dt.timedelta(int(365.25 * num_of_years))
start_date_str = dt.datetime.strftime(start_date, "%Y-%m-%d")
end_date = dt.datetime.now()
end_date_str = dt.datetime.strftime(end_date, "%Y-%m-%d")
print(f"\nStart Date: {start_date}")
print(f"End Date: {end_date}\n")

goog_df = retrieve_ohlcv_from_to(
    engine,
    exchange_code=exchange_code,
    ticker=ticker,
    start_date=start_date_str,
    end_date=end_date_str,
)

print(f"\nHead:\n{goog_df.head(3)}")


Start Date: 2016-04-26 12:02:02.715394
End Date: 2026-04-26 12:02:02.715452

Exchange_id is: 2

Head:
                  date     open     high      low    close    volume
Datetime                                                            
2018-10-26  2018-10-26  51.8515  55.3265  51.7045  53.5735  83751720
2018-10-29  2018-10-29  54.1235  54.8520  49.7915  51.0040  77614460
2018-10-30  2018-10-30  50.4230  51.8745  50.0375  51.8105  64253140


In [3]:
exchange_code = "ARCX"
ticker = "SPY"
num_of_years = 10
start_date = dt.datetime.now() - dt.timedelta(int(365.25 * num_of_years))
start_date_str = dt.datetime.strftime(start_date, "%Y-%m-%d")
end_date = dt.datetime.now()
end_date_str = dt.datetime.strftime(end_date, "%Y-%m-%d")
print(f"\nStart Date: {start_date}")
print(f"End Date: {end_date}\n")

spy_df = retrieve_ohlcv_from_to(
    engine,
    exchange_code=exchange_code,
    ticker=ticker,
    start_date=start_date_str,
    end_date=end_date_str,
)

print(f"\nHead:\n{spy_df.head(3)}")
print(f"\nTail:\n{spy_df.tail(3)}\n")


Start Date: 2016-04-26 12:02:06.487563
End Date: 2026-04-26 12:02:06.487622

Exchange_id is: 6

Head:
                  date    open    high     low   close     volume
Datetime                                                         
2018-10-26  2018-10-26  265.92  271.00  262.29  265.33  201855216
2018-10-29  2018-10-29  268.80  270.25  259.85  263.86  161040110
2018-10-30  2018-10-30  263.67  268.12  263.12  267.77  157427861

Tail:
                  date    open      high       low   close    volume
Datetime                                                            
2026-04-22  2026-04-22  709.15  711.4500  708.2200  711.21  42518498
2026-04-23  2026-04-23  709.50  712.3598  702.2803  708.45  56173991
2026-04-24  2026-04-24  710.75  714.4700  709.0100  713.94  45232458



In [4]:
# We'll use np.polyfit to fit a line to each rolling window
def rolling_slope(series, window_size=10): # Drop NaN values (incomplete windows)
    y = series.values
    x = np.arange(len(y))
    if len(y) < window_size or np.isnan(y).any():
        return np.nan
    slope, _ = np.polyfit(x, y, 1)  # degree=1 for linear fit
    return slope

In [5]:
# Calculate daily change
min_up_days_ratio = 0.6
window_size = 10
spy_df["daily_change"] = spy_df["close"].diff()
spy_df["up_day"] = spy_df["daily_change"] > 0
# Calculate percentage of up days
spy_df["up_days_ratio"] = spy_df["up_day"].rolling(window=window_size).mean()
print(f"\nUp Days Ratio (last {window_size} days): {spy_df['up_days_ratio'].iloc[-1]}\n")

# Calculate slope of trendline (positive slope means uptrend)
# Step 1: Calculate moving average (window=3 for example) window_size = 3
spy_df['close_ma'] = spy_df['close'].rolling(window=window_size).mean()
spy_df["ma_slope"] = spy_df["close_ma"].rolling(window=window_size).apply(rolling_slope, args=(window_size,), raw=False)
# Determine if consistently moving higher
spy_df["consistently_moving_higher"] = (spy_df["ma_slope"] > 0) & (
    spy_df["up_days_ratio"] >= min_up_days_ratio)
print(f"\nTail:\n{spy_df.tail(20)}")


Up Days Ratio (last 10 days): 0.7


Tail:
                  date    open      high       low   close     volume  \
Datetime                                                                
2026-03-27  2026-03-27  642.50  642.6600  633.1050  634.09  103649369   
2026-03-30  2026-03-30  640.11  640.3700  629.2800  631.97   99275851   
2026-03-31  2026-03-31  638.94  651.5400  637.9800  650.34  152534103   
2026-04-01  2026-04-01  653.90  658.5200  653.0000  655.24   97841472   
2026-04-02  2026-04-02  646.42  658.2000  645.1100  655.83   68358714   
2026-04-06  2026-04-06  655.86  659.7200  655.5200  658.93   39105759   
2026-04-07  2026-04-07  656.65  659.6100  651.0600  659.22   69980362   
2026-04-08  2026-04-08  676.39  677.0800  671.4600  676.01   93606115   
2026-04-09  2026-04-09  674.84  681.1600  673.7700  679.91   57119363   
2026-04-10  2026-04-10  681.32  682.0300  678.4500  679.46   43024242   
2026-04-13  2026-04-13  677.41  686.3000  676.5800  686.10   54353487   
2026-04-

In [6]:
spy_df['close_ma'] = spy_df['close'].rolling(window=window_size).mean()
spy_df["ma_slope"] = spy_df["close_ma"].rolling(window=window_size).apply(rolling_slope, raw=False)

print(f"\nTail:\n{spy_df.tail(3)}")


Tail:
                  date    open      high       low   close    volume  \
Datetime                                                               
2026-04-22  2026-04-22  709.15  711.4500  708.2200  711.21  42518498   
2026-04-23  2026-04-23  709.50  712.3598  702.2803  708.45  56173991   
2026-04-24  2026-04-24  710.75  714.4700  709.0100  713.94  45232458   

            daily_change  up_day  up_days_ratio  close_ma  ma_slope  \
Datetime                                                              
2026-04-22          7.13    True            0.7   697.568  4.962588   
2026-04-23         -2.76   False            0.6   700.422  4.811176   
2026-04-24          5.49    True            0.7   703.870  4.522315   

            consistently_moving_higher  
Datetime                                
2026-04-22                        True  
2026-04-23                        True  
2026-04-24                        True  


In [8]:
df = pd.merge(goog_df, spy_df, on="date", how="inner", suffixes=(None, "_spy"))
print(f"\nHead:\n{df.head(3)}")


Head:
         date     open     high      low    close    volume  open_spy  \
0  2018-10-26  51.8515  55.3265  51.7045  53.5735  83751720    265.92   
1  2018-10-29  54.1235  54.8520  49.7915  51.0040  77614460    268.80   
2  2018-10-30  50.4230  51.8745  50.0375  51.8105  64253140    263.67   

   high_spy  low_spy  close_spy  volume_spy  daily_change  up_day  \
0    271.00   262.29     265.33   201855216           NaN   False   
1    270.25   259.85     263.86   161040110         -1.47   False   
2    268.12   263.12     267.77   157427861          3.91    True   

   up_days_ratio  close_ma  ma_slope  consistently_moving_higher  
0            NaN       NaN       NaN                       False  
1            NaN       NaN       NaN                       False  
2            NaN       NaN       NaN                       False  


In [9]:

# Dictionary mapping old column names to new ones
rename_map = {
    'date': 'Date',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
}
# Rename columns
df.rename(columns=rename_map, inplace=True)
print(f"\nHead after renaming columns:\n{df.head(3)}")


Head after renaming columns:
         Date     Open     High      Low    Close    Volume  open_spy  \
0  2018-10-26  51.8515  55.3265  51.7045  53.5735  83751720    265.92   
1  2018-10-29  54.1235  54.8520  49.7915  51.0040  77614460    268.80   
2  2018-10-30  50.4230  51.8745  50.0375  51.8105  64253140    263.67   

   high_spy  low_spy  close_spy  volume_spy  daily_change  up_day  \
0    271.00   262.29     265.33   201855216           NaN   False   
1    270.25   259.85     263.86   161040110         -1.47   False   
2    268.12   263.12     267.77   157427861          3.91    True   

   up_days_ratio  close_ma  ma_slope  consistently_moving_higher  
0            NaN       NaN       NaN                       False  
1            NaN       NaN       NaN                       False  
2            NaN       NaN       NaN                       False  
